# Query a Neo4j knowledge graph

This notebook runs graph-only retrieval and answer generation over an existing Neo4j knowledge-graph collection. Run the knowledge-graph creation notebook first.

## Prerequisites

Set `MAAS_API_KEY`, `MAAS_BASE_URL`, `NEO4J_URI`, and `NEO4J_PASSWORD`. These Neo4j settings and the collection name must match the graph-creation notebook.

In [ ]:
%pip install 'ai4rag~={AI4RAG_VERSION}' | tail -n 1

## Configure MaaS and initialize models

In [ ]:
import getpass
import os

from ai4rag.rag.embedding.openai_model import OpenAIEmbeddingModel, OpenAIEmbeddingParams
from ai4rag.rag.foundation_models.base_model import Language
from ai4rag.rag.foundation_models.openai_model import OpenAIFoundationModel
from ai4rag.utils.clients.maas_client import create_maas_client

maas_api_key = os.getenv("MAAS_API_KEY") or getpass.getpass("MAAS_API_KEY: ")
maas_base_url = os.getenv("MAAS_BASE_URL") or getpass.getpass("MAAS_BASE_URL: ")
client = create_maas_client(base_url=maas_base_url, api_key=maas_api_key)

embedding_model = OpenAIEmbeddingModel(
    client=client,
    model_id="{EMBEDDING_MODEL_ID}",
    params=OpenAIEmbeddingParams(**{EMBEDDING_PARAMS}),
)
foundation_model = OpenAIFoundationModel(
    client=client,
    model_id="{FM_MODEL_ID}",
    system_message_text="""{SYSTEM_MESSAGE}""",
    user_message_text="""{USER_MESSAGE}""",
    context_template_text="""{CONTEXT_TEXT}""",
    language=Language(**{LANGUAGE}),
)

## Connect to the Neo4j graph

`Neo4jGraphStore` uses a collection-specific vector index to select seed chunks, then expands them through sequential `NEXT_CHUNK` and entity relationships.

In [ ]:
from ai4rag.rag.vector_store.config import Neo4jConfig
from ai4rag.rag.vector_store.neo4j import Neo4jGraphStore

collection_name = "{COLLECTION_NAME}"
vector_store = Neo4jGraphStore(
    embedding_model=embedding_model,
    config=Neo4jConfig.from_env(),
    collection_name=collection_name,
)

## Configure graph retrieval and generate an answer

The current `Retriever` exposes the graph mode and number of seed chunks. Its graph-expansion defaults are one sequential hop in each direction, entity-neighbor expansion enabled, and at most five entity-linked chunks per seed.

In [ ]:
from ai4rag.rag.retrieval.retriever import Retriever
from ai4rag.rag.template.simple_rag_template import SimpleRAG

retriever = Retriever(
    vector_store=vector_store,
    method="simple",
    number_of_chunks={NUMBER_OF_CHUNKS},
    search_mode="graph",
)
rag_pattern = SimpleRAG(foundation_model=foundation_model, retriever=retriever)

question = input("Question: ")
response = rag_pattern.generate(question=question)
print(response["answer"])

print("\nRetrieved graph contexts:")
for index, chunk in enumerate(response["reference_documents"], start=1):
    print(f"\n--- Context {{index}} ---\n{{chunk.text}}")

## Direct graph-search controls

For inspection outside the RAG pattern, call `vector_store.search()` directly. `graph_hops` must be positive; `entity_neighbor_limit=0` suppresses entity-neighbor text. Neo4j accepts only `search_mode="graph"`.

In [ ]:
results = vector_store.search(
    query=question,
    k={NUMBER_OF_CHUNKS},
    search_mode="graph",
    graph_hops=1,
    include_entity_neighbors=True,
    entity_neighbor_limit=5,
    include_scores=True,
)
for chunk, score in results:
    print(f"score={{score:.4f}} document_id={{chunk.metadata.get('document_id', '')}}")
    print(chunk.text[:500])

In [ ]:
vector_store.close()